# Demo — GNN-BERT for Music Context Understanding

**CSE715 · Tanvir Rahman (22241134)**

A walkthrough of all four tasks and what they actually showed. Everything here
reads from committed artefacts in `results/`, so the notebook runs in seconds
with no GPU and no audio.

**The short version:** all four tasks work end to end, but cross-modal fusion
does *not* reliably beat its stronger unimodal branch. The interesting part is
that we can say precisely why.

In [1]:
import json, sys
from pathlib import Path
sys.path.insert(0, "..")
R = Path("../results")
def load(n): return json.load(open(R / n))

print("available artefacts:")
for f in sorted(R.glob("*.json")): print("  ", f.name)

available artefacts:
   case_studies_task3.json
   examples_task1_mtat.json
   examples_task1_musiccaps.json
   examples_task1_musiccaps_stripped.json
   metrics.json
   metrics_task1_mtat.json
   metrics_task1_musiccaps.json
   metrics_task1_musiccaps_stripped.json
   metrics_task2.json
   metrics_task2_b4.json
   metrics_task3.json
   metrics_task3_multitask.json
   metrics_task4.json
   metrics_task4_zeroshot.json
   tsne_task3.json


## Task 1 — BERT tag classifier

Two dataset variants, both sanctioned by the specification. Each has a defect
the other does not, so we ran both.

In [2]:
rows = [("MagnaTagATune (metadata)", "metrics_task1_mtat.json"),
        ("MusicCaps (raw captions)", "metrics_task1_musiccaps.json"),
        ("MusicCaps (stripped)",     "metrics_task1_musiccaps_stripped.json")]
print(f"{'setting':<28}{'micro-F1':>10}{'macro-F1':>10}{'best baseline':>28}")
print("-" * 76)
for label, f in rows:
    d = load(f)
    b = d["baselines"]
    name, best = max(((k, v["micro_f1"]) for k, v in b.items()), key=lambda kv: kv[1])
    print(f"{label:<28}{d['test']['micro_f1']:>10.3f}{d['test']['macro_f1']:>10.3f}"
          f"{name + ' ' + format(best, '.3f'):>28}")
print()
print("Oracle ceiling for ANY text-only model on MagnaTagATune: 0.673 / 0.575")

setting                       micro-F1  macro-F1               best baseline
----------------------------------------------------------------------------
MagnaTagATune (metadata)         0.275     0.192  B1_random_prevalence 0.103
MusicCaps (raw captions)         0.565     0.531      B5_lexical_match 0.540
MusicCaps (stripped)             0.488     0.438  B1_random_prevalence 0.089

Oracle ceiling for ANY text-only model on MagnaTagATune: 0.673 / 0.575


**Read this carefully.** The raw-caption number (0.687) looks best, but a pure
substring matcher already scores **0.581** on that task — 52% of MusicCaps
aspects appear verbatim in their own caption. Only ~0.105 of the gap is language
understanding.

The **stripped** variant is the honest headline: the label words are deleted
from the input, the lexical baseline drops to exactly **0.000**, and the model
still reaches **0.608**.

In [3]:
d = load("examples_task1_musiccaps_stripped.json")[0]
print("TRUE aspects:", ", ".join(d["true"]))
print("\nINPUT (label words deleted):")
print(" ", d["text"][:230], "...")
print("\nMODEL PREDICTIONS:")
for p in d["predicted"][:6]:
    mark = "<-- recovered" if p["tag"] in d["true"] else ""
    print(f"  {p['tag']:<22}{p['p']:.3f}  {mark}")

TRUE aspects: low quality

INPUT (label words deleted):
  The recording features a ballad song that contains sustained strings, mellow piano melody and soft female vocal singing over it. It sounds sad and soulful, like something you would hear at Sunday serv ...

MODEL PREDICTIONS:
  low quality           0.838  <-- recovered
  passionate            0.734  
  emotional             0.700  
  noisy                 0.575  
  flat male vocal       0.541  
  mellow                0.457  


The phrases *low quality*, *groovy bass*, *punchy kick* and *punchy snare* are
absent from the input, yet all four are predicted with high confidence — inferred
from surrounding descriptive context (steel drums, brass stabs, crash cymbals)
rather than detected as strings.

## Task 2 — GNN on music structure graphs vs. a CNN

In [4]:
t2 = load("metrics_task2.json")
print(f"FMA-small, {len(t2['genres'])} balanced genres (chance = {1/len(t2['genres']):.3f})")
print(f"artist leakage, VERIFIED not assumed: {t2['artist_leakage']}")
print()
print(f"{'model':<12}{'params':>10}{'accuracy':>10}{'macro-F1':>10}")
print("-" * 42)
for m, r in t2["runs"].items():
    print(f"{r['model']:<12}{r['params']:>10,}{r['test_accuracy']:>10.3f}{r['test_macro_f1']:>10.3f}")
gat, cnn = t2["runs"]["gat"], t2["runs"]["cnn"]
print(f"\nGAT matches the CNN on macro-F1 with {cnn['params']/gat['params']:.1f}x fewer parameters,")
print("but the CNN wins on accuracy. Structure is competitive, not superior.")

FMA-small, 8 balanced genres (chance = 0.125)
artist leakage, VERIFIED not assumed: {'train_test': 0, 'train_val': 0, 'val_test': 0}

model           params  accuracy  macro-F1
------------------------------------------
sage            49,416     0.396     0.401
gat             26,120     0.432     0.426
cnn_b2         241,992     0.482     0.479

GAT matches the CNN on macro-F1 with 9.3x fewer parameters,
but the CNN wins on accuracy. Structure is competitive, not superior.


In [5]:
gat_f1 = t2["runs"]["gat"]["per_class_f1"]
print("per-genre F1 (GAT), hardest last:")
for g, f in sorted(gat_f1.items(), key=lambda kv: -kv[1]):
    bar = "#" * int(f * 40)
    print(f"  {g:<15}{f:.3f}  {bar}")
print("\n'Experimental' is a residual category defined by what it is NOT,")
print("so there is no consistent acoustic signature to learn.")

per-genre F1 (GAT), hardest last:
  Hip-Hop        0.686  ###########################
  Rock           0.589  #######################
  Electronic     0.509  ####################
  International  0.493  ###################
  Instrumental   0.374  ##############
  Folk           0.304  ############
  Pop            0.281  ###########
  Experimental   0.172  ######

'Experimental' is a residual category defined by what it is NOT,
so there is no consistent acoustic signature to learn.


## Task 3 — GNN-BERT fusion, and why it underperforms

In [6]:
t3 = load("metrics_task3.json")
print(f"{'mode':<14}{'params':>12}{'macro-F1':>10}{'micro-F1':>10}{'AUC-PR':>9}")
print("-" * 55)
for m, r in t3["runs"].items():
    print(f"{m:<14}{r['params']:>12,}{r['test_macro_f1']:>10.3f}"
          f"{r['test_micro_f1']:>10.3f}{r['test_auc_pr']:>9.3f}")
b, c = t3["runs"]["bert"], t3["runs"]["crossattn"]
print(f"\ncross-attention vs BERT-only:")
print(f"  macro-F1 {c['test_macro_f1']-b['test_macro_f1']:+.3f}   "
      f"micro-F1 {c['test_micro_f1']-b['test_micro_f1']:+.3f}   "
      f"AUC-PR {c['test_auc_pr']-b['test_auc_pr']:+.3f}")
print("\nFusion wins on micro-F1 and AUC-PR but LOSES on macro-F1.")
print("These margins do not establish the project's central claim.")

mode                params  macro-F1  micro-F1   AUC-PR
-------------------------------------------------------
bert            66,402,868     0.171     0.306    0.168
gnn                 31,796     0.123     0.124    0.084
concat          66,434,612     0.168     0.314    0.182
crossattn       66,834,740     0.192     0.324    0.176

cross-attention vs BERT-only:
  macro-F1 +0.022   micro-F1 +0.019   AUC-PR +0.009

Fusion wins on micro-F1 and AUC-PR but LOSES on macro-F1.
These margins do not establish the project's central claim.


### Why: the attention collapsed

If cross-attention were selecting informative caption tokens, the weights would
be peaked. They are not.

In [7]:
import math
cases = load("case_studies_task3.json")
print(f"{'clip':>7}{'max w':>9}{'min w':>9}{'ratio':>8}{'entropy % of uniform':>24}")
print("-" * 58)
for cs in cases:
    w = [t["weight"] for t in cs["attention_top_tokens"]]
    H = -sum(x * math.log(x) for x in w if x > 0)
    print(f"{cs['clip_id']:>7}{max(w):>9.4f}{min(w):>9.4f}"
          f"{max(w)/min(w):>8.2f}{H/math.log(len(w))*100:>23.1f}%")
print("\nA ratio near 1.0 means every token gets equal weight: the graph query")
print("is not discriminating, so A*H_text degenerates to a mean over the text")
print("and the model reduces to early concat -- exactly what the table shows.")

   clip    max w    min w   ratio    entropy % of uniform
----------------------------------------------------------
     14   0.0927   0.0908    1.02                   84.2%
     90   0.0573   0.0557    1.03                   62.4%
    177   0.0938   0.0900    1.04                   84.2%

A ratio near 1.0 means every token gets equal weight: the graph query
is not discriminating, so A*H_text degenerates to a mean over the text
and the model reduces to early concat -- exactly what the table shows.


In [8]:
for cs in cases:
    print(f"clip {cs['clip_id']}: {cs['text']}")
    print(f"   graph : {cs['graph']['n_nodes']} nodes, {cs['graph']['n_edges']} edges, "
          f"busiest segment #{cs['graph']['busiest_segment']}")
    print(f"   true  : {', '.join(cs['true_tags'])}")
    print(f"   pred  : {', '.join(p['tag'] for p in cs['top_predictions'][:4])}")
    print()

clip 14: Contimune | LVX Nova
   graph : 19 nodes, 70 edges, busiest segment #16
   true  : techno
   pred  : techno, electronic, beat, drum

clip 90: My mistress hath a pritty thing (Tobias Hume) | Gambomania
   graph : 19 nodes, 60 edges, busiest segment #1
   true  : cello, classical, fast, solo, string, violin
   pred  : vocal, female vocal, opera, male vocal

clip 177: 7 Nine Skank | Stereo Mash Up
   graph : 19 nodes, 78 edges, busiest segment #7
   true  : drum, electronic, synth, weird
   pred  : techno, electronic, fast, beat



### The DEAM auxiliary term: a measurable trade-off

In [9]:
mt = load("metrics_task3_multitask.json")["runs"]["crossattn"]
base = t3["runs"]["crossattn"]
print(f"{'':<12}{'tags only':>12}{'+ DEAM':>10}{'delta':>10}")
print("-" * 44)
for k, lab in (("test_macro_f1","macro-F1"), ("test_micro_f1","micro-F1"),
               ("test_auc_pr","AUC-PR")):
    print(f"{lab:<12}{base[k]:>12.3f}{mt[k]:>10.3f}{mt[k]-base[k]:>+10.3f}")
print()
for k, sd in (("valence", 1.174), ("arousal", 1.282)):
    m = mt[k]
    print(f"{k:<9} MAE {m['mae']:.3f} (z) = {m['mae']*sd:.2f} on the 1-9 scale   R2 {m['r2']:+.3f}")
print("\nBoth R2 are positive, so the emotion heads beat predicting the mean.")
print("But tag macro-F1 falls 36% relative. Training data is identical across")
print("the two runs, so this is the auxiliary term competing for capacity.")

               tags only    + DEAM     delta
--------------------------------------------
macro-F1           0.192     0.116    -0.076
micro-F1           0.324     0.279    -0.046
AUC-PR             0.176     0.132    -0.044

valence   MAE 0.710 (z) = 0.83 on the 1-9 scale   R2 +0.181
arousal   MAE 0.721 (z) = 0.92 on the 1-9 scale   R2 +0.085

Both R2 are positive, so the emotion heads beat predicting the mean.
But tag macro-F1 falls 36% relative. Training data is identical across
the two runs, so this is the auxiliary term competing for capacity.


## Task 4 — contrastive caption/audio retrieval

In [10]:
t4 = load("metrics_task4.json")
g = t4["gallery_size"]
print(f"gallery: {g} clips   (R@K is meaningless without this)")
print(f"\n{'direction':<20}{'R@1':>9}{'R@5':>9}{'R@10':>9}")
print("-" * 47)
for k, lab in (("caption_to_audio","caption -> audio"), ("audio_to_caption","audio -> caption")):
    r = t4[k]; print(f"{lab:<20}{r['R@1']:>9.4f}{r['R@5']:>9.4f}{r['R@10']:>9.4f}")
print(f"{'random baseline':<20}{t4['random_baseline_R@1']:>9.4f}{'':>9}{t4['random_baseline_R@10']:>9.4f}")
print(f"\nvs random: {t4['caption_to_audio']['R@1']/t4['random_baseline_R@1']:.1f}x at R@1, "
      f"{t4['caption_to_audio']['R@10']/t4['random_baseline_R@10']:.1f}x at R@10")
print(f"median rank of the true clip: {t4['median_rank_caption_to_audio']:.0f} / {g}")

gallery: 2773 clips   (R@K is meaningless without this)

direction                 R@1      R@5     R@10
-----------------------------------------------
caption -> audio       0.0029   0.0108   0.0220
audio -> caption       0.0022   0.0126   0.0224
random baseline        0.0004            0.0036

vs random: 8.0x at R@1, 6.1x at R@10
median rank of the true clip: 527 / 2773


In [11]:
ex = load("retrieval_examples/task4_examples.json")
print(f"true-clip ranks across {len(ex)} queries: {[e['rank_of_true'] for e in ex]}")
print(f"rank-1 hits: {sum(1 for e in ex if e['top3'][0]['is_correct'])}/{len(ex)}")
print()
q = ex[0]
print("QUERY CAPTION:"); print(" ", q["query_caption"][:200], "...")
print(f"\ntrue clip ranked #{q['rank_of_true']}. Top-3 retrieved:")
for i, t in enumerate(q["top3"], 1):
    print(f"  {i}. score {t['score']:.3f} {'[CORRECT]' if t['is_correct'] else ''}")
    print(f"     {t['caption'][:110]}...")

true-clip ranks across 10 queries: [29, 512, 832, 1840, 315, 288, 1256, 66, 10, 1001]
rank-1 hits: 0/10

QUERY CAPTION:
  The low quality recording features a ballad song that contains sustained strings, mellow piano melody and soft female vocal singing over it. It sounds sad and soulful, like something you would hear at ...

true clip ranked #29. Top-3 retrieved:
  1. score 0.494 
     The song is instrumental music. The tempo is medium tempo with a piano bass note playing rhythmically, along w...
  2. score 0.489 
     This is an outro for a kids song. The song features quirky male vocals. Right after the end of the phrase, thu...
  3. score 0.485 
     This music is an instrumental with a melancholic piano melody, mesmerising sound of chimes, simple keyboard ha...


The alignment is real (9x random at R@1) but far from usable retrieval — no
query places its true clip first. The model learns coarse
acoustic-descriptive correspondence (recording quality, instrumentation family,
tempo band) without fine discrimination.

In [12]:
zs = load("metrics_task4_zeroshot.json")
sup = load("metrics_task1_musiccaps_stripped.json")["test"]["micro_f1"]
print("zero-shot tagging from the contrastive space (NO tag supervision):")
print(f"  micro-F1 {zs['micro_f1']:.4f}   macro-F1 {zs['macro_f1']:.4f}   AUC-PR {zs['auc_pr']:.4f}")
print(f"\nTask 1 SUPERVISED on the same corpus: micro-F1 {sup:.4f}")
print(f"-> alignment alone recovers {zs['micro_f1']/sup*100:.0f}% of supervised performance")

zero-shot tagging from the contrastive space (NO tag supervision):
  micro-F1 0.1070   macro-F1 0.0954   AUC-PR 0.0621

Task 1 SUPERVISED on the same corpus: micro-F1 0.4877
-> alignment alone recovers 22% of supervised performance


## Summary across all four tasks

In [13]:
print(f"{'task':<38}{'metric':<22}{'value':>9}{'baseline':>11}")
print("-" * 80)
r1 = load("metrics_task1_musiccaps_stripped.json")
print(f"{'1  BERT tagger (MusicCaps stripped)':<38}{'micro-F1':<22}{r1['test']['micro_f1']:>9.3f}{0.0:>11.3f}")
r1b = load("metrics_task1_mtat.json")
print(f"{'1  BERT tagger (MagnaTagATune)':<38}{'micro-F1':<22}{r1b['test']['micro_f1']:>9.3f}"
      f"{r1b['baselines']['B1_random_prevalence']['micro_f1']:>11.3f}")
print(f"{'2  GAT on structure graphs':<38}{'macro-F1':<22}{t2['runs']['gat']['test_macro_f1']:>9.3f}{0.125:>11.3f}")
print(f"{'2  CNN baseline (B2)':<38}{'macro-F1':<22}{t2['runs']['cnn']['test_macro_f1']:>9.3f}{0.125:>11.3f}")
print(f"{'3  cross-attention fusion':<38}{'macro-F1':<22}{c['test_macro_f1']:>9.3f}"
      f"{b['test_macro_f1']:>11.3f}")
print(f"{'3  + DEAM emotion (valence)':<38}{'R2':<22}{mt['valence']['r2']:>9.3f}{0.0:>11.3f}")
print(f"{'4  retrieval (caption->audio)':<38}{'R@10':<22}"
      f"{t4['caption_to_audio']['R@10']:>9.3f}{t4['random_baseline_R@10']:>11.3f}")
print(f"{'4  zero-shot tagging':<38}{'micro-F1':<22}{zs['micro_f1']:>9.3f}{sup:>11.3f}")
print()
print("Every result beats its baseline except Task 3's fusion, which does not")
print("beat its own BERT-only arm on macro-F1. That negative result, and its")
print("diagnosis, is the main finding of the project.")

task                                  metric                    value   baseline
--------------------------------------------------------------------------------
1  BERT tagger (MusicCaps stripped)   micro-F1                  0.488      0.000
1  BERT tagger (MagnaTagATune)        micro-F1                  0.275      0.103
2  GAT on structure graphs            macro-F1                  0.426      0.125
2  CNN baseline (B2)                  macro-F1                  0.479      0.125
3  cross-attention fusion             macro-F1                  0.192      0.171
3  + DEAM emotion (valence)           R2                        0.181      0.000
4  retrieval (caption->audio)         R@10                      0.022      0.004
4  zero-shot tagging                  micro-F1                  0.107      0.488

Every result beats its baseline except Task 3's fusion, which does not
beat its own BERT-only arm on macro-F1. That negative result, and its
diagnosis, is the main finding of the project.
